## RNN

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [20]:
# 数据预处理：将图像转换为张量并进行归一化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # 归一化到 [0, 1] 范围
])

# 加载训练集和测试集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [21]:
def train(model, train_loader, criterion, optimizer, num_epochs=5):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            # 将图像调整为 (batch_size, sequence_length, input_size) 形式
            images = images.view(-1, 28, 28).to(device)  # 每张图片的28行作为时间步

            # 清零梯度
            optimizer.zero_grad()

            # 前向传播
            outputs = model(images)

            # 计算损失
            loss = criterion(outputs, labels.to(device))

            # 反向传播并优化
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # 计算准确率
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels.to(device)).sum().item()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100 * correct / total:.2f}%")


In [22]:
def test(model, test_loader):
    model.eval()  # 设置为评估模式
    correct = 0
    total = 0
    with torch.no_grad():  # 不计算梯度
        for images, labels in test_loader:
            images = images.view(-1, 28, 28).to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels.to(device)).sum().item()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")


# RNN

## 手动

In [1]:
import numpy as np

# =========================
#  1. 超参数设置
# =========================

# 序列长度 (为了演示，假设我们用固定长度训练)
T = 5                    # 每个序列包含的时间步数
d_x = 3                  # 输入维度
d_h = 4                  # 隐层维度
d_y = 3                  # 输出维度

learning_rate = 1e-2     # 学习率
num_epochs = 2000        # 训练迭代次数
np.random.seed(42)       # 随机种子，方便复现

# =========================
#  2. 构造数据集
# =========================

# 这里构造一个简单的数据：输入和输出是相同的序列
# X.shape = (batch_size, T, d_x)
# Y.shape = (batch_size, T, d_y)

# 为了演示，这里batch_size=1（单条序列训练），也可以改成更大的batch_size
batch_size = 1
X = np.random.randn(batch_size, T, d_x)
Y = X.copy()  # 目标是输出与输入相同

# =========================
#  3. 初始化 RNN 参数
# =========================

# W_{xh} : (d_h, d_x)
W_xh = np.random.randn(d_h, d_x) * 0.01
# W_{hh} : (d_h, d_h)
W_hh = np.random.randn(d_h, d_h) * 0.01
# b_h : (d_h,)
b_h = np.zeros((d_h,))

# W_{hy} : (d_y, d_h)
W_hy = np.random.randn(d_y, d_h) * 0.01
# b_y : (d_y,)
b_y = np.zeros((d_y,))

# 方便打包到一起，后面更新时方便处理
params = (W_xh, W_hh, b_h, W_hy, b_y)

# =========================
#  4. 前向传播函数
# =========================
def rnn_forward(x_seq, params, h0=None):
    """
    x_seq: shape (T, d_x)
    params: (W_xh, W_hh, b_h, W_hy, b_y)
    h0: shape (d_h,)，初始隐状态。若为None，则默认0向量。
    返回：
        hs: list，保存每个时刻的隐状态 (T, d_h)
        ys: list，保存每个时刻的输出 (T, d_y)
    """
    W_xh, W_hh, b_h, W_hy, b_y = params
    
    T = x_seq.shape[0]
    d_h = W_hh.shape[0]
    if h0 is None:
        h_prev = np.zeros(d_h)
    else:
        h_prev = h0
    
    hs = []
    ys = []
    
    for t in range(T):
        x_t = x_seq[t]  # shape (d_x, )
        
        # h_t = tanh(W_xh*x_t + W_hh*h_{t-1} + b_h)
        h_t = np.tanh(W_xh @ x_t + W_hh @ h_prev + b_h)
        
        # y_t = W_hy*h_t + b_y
        y_t = W_hy @ h_t + b_y
        
        hs.append(h_t)
        ys.append(y_t)
        
        h_prev = h_t  # 更新隐状态
    
    # 转为 np.array 便于后续操作
    hs = np.stack(hs, axis=0)  # (T, d_h)
    ys = np.stack(ys, axis=0)  # (T, d_y)
    
    return hs, ys

# =========================
#  5. 反向传播 (BPTT)
# =========================

def rnn_backward(x_seq, hs, ys, y_true, params):
    """
    x_seq: shape (T, d_x)
    hs: shape (T, d_h)
    ys: shape (T, d_y)
    y_true: shape (T, d_y)
    params: (W_xh, W_hh, b_h, W_hy, b_y)
    
    返回：
        grads: (dW_xh, dW_hh, db_h, dW_hy, db_y)
    """
    W_xh, W_hh, b_h, W_hy, b_y = params
    T = x_seq.shape[0]
    d_h = W_hh.shape[0]
    
    # 初始化梯度
    dW_xh = np.zeros_like(W_xh)
    dW_hh = np.zeros_like(W_hh)
    db_h  = np.zeros_like(b_h)
    dW_hy = np.zeros_like(W_hy)
    db_y  = np.zeros_like(b_y)
    
    # 对隐状态的梯度进行累加 (通过时间反向传播)
    dh_next = np.zeros(d_h)  # 下一个时刻传回来的梯度
    
    # 从最后一个时刻往前反传
    for t in reversed(range(T)):
        y_pred = ys[t]        # (d_y, )
        y_true_t = y_true[t]  # (d_y, )
        h_t = hs[t]           # (d_h, )
        
        # ========== 对输出层 ========== 
        # 损失函数: MSE = 0.5 * sum((y_pred - y_true)^2)
        # dL/dy_pred = (y_pred - y_true)
        dy = (y_pred - y_true_t)
        
        # dW_hy += dy * h_t^T
        dW_hy += np.outer(dy, h_t)
        # db_y += dy
        db_y += dy
        
        # 接下来对隐状态的梯度
        # dh = W_hy^T * dy + dh_next
        dh = W_hy.T @ dy + dh_next
        
        # ========== 对隐层（tanh）的梯度 ========== 
        # h_t = tanh(z_t), z_t = W_xh x_t + W_hh h_{t-1} + b_h
        # d(tanh(z)) = (1 - h_t^2)
        dz = dh * (1 - h_t**2)  # element-wise
        
        # ========== 对各权重及偏置 ========== 
        # dW_xh += dz * x_t^T
        x_t = x_seq[t]
        dW_xh += np.outer(dz, x_t)
        
        # dW_hh += dz * h_{t-1}^T
        h_prev = hs[t-1] if t > 0 else np.zeros_like(h_t)
        dW_hh += np.outer(dz, h_prev)
        
        # db_h += dz
        db_h += dz
        
        # ========== 传递给上一时刻的隐状态梯度 ==========
        # dh_next = W_hh^T * dz
        dh_next = W_hh.T @ dz
    
    grads = (dW_xh, dW_hh, db_h, dW_hy, db_y)
    return grads

# =========================
#  6. 训练循环
# =========================

for epoch in range(num_epochs):
    # 1) 前向传播
    # 我们的batch_size=1，只取 X[0], Y[0] 做演示
    x_seq = X[0]  # (T, d_x)
    y_seq = Y[0]  # (T, d_y)
    
    hs, ys = rnn_forward(x_seq, params)
    
    # 2) 计算当前的损失（简单使用 MSE）
    loss = 0.5 * np.sum((ys - y_seq)**2)
    
    # 3) 反向传播，计算梯度
    grads = rnn_backward(x_seq, hs, ys, y_seq, params)
    
    # 4) 参数更新
    W_xh, W_hh, b_h, W_hy, b_y = params
    dW_xh, dW_hh, db_h, dW_hy, db_y = grads
    
    W_xh -= learning_rate * dW_xh
    W_hh -= learning_rate * dW_hh
    b_h  -= learning_rate * db_h
    W_hy -= learning_rate * dW_hy
    b_y  -= learning_rate * db_y
    
    params = (W_xh, W_hh, b_h, W_hy, b_y)
    
    # 每隔一段打印一次损失
    if (epoch+1) % 200 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss = {loss:.4f}")

# =========================
#  7. 测试
# =========================
# 经过训练后，我们看看网络的预测表现
hs_test, ys_test = rnn_forward(X[0], params)
print("Final predictions:")
print(ys_test)
print("Ground truth:")
print(Y[0])


Epoch [200/2000], Loss = 0.7753
Epoch [400/2000], Loss = 0.1431
Epoch [600/2000], Loss = 0.1259
Epoch [800/2000], Loss = 0.0752
Epoch [1000/2000], Loss = 0.0210
Epoch [1200/2000], Loss = 0.0074
Epoch [1400/2000], Loss = 0.0026
Epoch [1600/2000], Loss = 0.0008
Epoch [1800/2000], Loss = 0.0002
Epoch [2000/2000], Loss = 0.0001
Final predictions:
[[ 0.50033224 -0.140034    0.64982752]
 [ 1.51676673 -0.23214619 -0.23674936]
 [ 1.58380143  0.76534908 -0.46718015]
 [ 0.5394877  -0.46098943 -0.4673104 ]
 [ 0.24531172 -1.91456569 -1.72380937]]
Ground truth:
[[ 0.49671415 -0.1382643   0.64768854]
 [ 1.52302986 -0.23415337 -0.23413696]
 [ 1.57921282  0.76743473 -0.46947439]
 [ 0.54256004 -0.46341769 -0.46572975]
 [ 0.24196227 -1.91328024 -1.72491783]]


## 调用框架

In [23]:
# 2. 定义RNN模型
class SimpleRNN(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, output_size=10, num_layers=1):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # 初始化隐藏层状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # 通过RNN进行前向传播
        out, _ = self.rnn(x, h0)
        
        # 只取RNN最后时刻的输出
        out = self.fc(out[:, -1, :])  # (batch_size, hidden_size) -> (batch_size, output_size)
        return out

In [26]:
# 4. 模型训练与测试
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 初始化模型、损失函数、优化器
model = SimpleRNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练和评估模型
train(model, train_loader, criterion, optimizer,num_epochs = 5)
accuracy = test(model, test_loader)


Epoch [1/5], Loss: 0.7569, Accuracy: 74.67%
Epoch [2/5], Loss: 0.3373, Accuracy: 90.31%
Epoch [3/5], Loss: 0.2389, Accuracy: 93.29%
Epoch [4/5], Loss: 0.1976, Accuracy: 94.49%
Epoch [5/5], Loss: 0.1790, Accuracy: 94.97%
Test Accuracy: 95.99%


# LSTM

In [16]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, output_size=10, num_layers=2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # 定义 LSTM 层
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # 定义全连接层
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # 初始化隐藏状态和细胞状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # LSTM 输出
        out, _ = self.lstm(x, (h0, c0))

        # 只取序列的最后一个时间步的输出
        out = self.fc(out[:, -1, :])
        return out


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 初始化 LSTM 模型
model = LSTMModel().to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
train(model, train_loader, criterion, optimizer, num_epochs=5)

# 测试模型
test(model, test_loader)


Epoch [1/5], Loss: 0.3528, Accuracy: 88.54%
Epoch [2/5], Loss: 0.0926, Accuracy: 97.17%
Epoch [3/5], Loss: 0.0641, Accuracy: 98.08%
Epoch [4/5], Loss: 0.0492, Accuracy: 98.51%
Epoch [5/5], Loss: 0.0418, Accuracy: 98.71%
Test Accuracy: 98.75%


# GRU

In [12]:
class GRUModel(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, output_size=10, num_layers=2):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # 定义 GRU 层
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)

        # 定义全连接层
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # 初始化隐藏状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # GRU 前向传播
        out, _ = self.gru(x, h0)

        # 取最后时间步的输出
        out = self.fc(out[:, -1, :])  # 输出形状: (batch_size, output_size)
        return out


In [19]:
# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 初始化 GRU 模型
model = GRUModel().to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练 GRU 模型
train(model, train_loader, criterion, optimizer, num_epochs=5)

# 测试 GRU 模型
test(model, test_loader)

Epoch [1/5], Loss: 0.3387, Accuracy: 88.99%
Epoch [2/5], Loss: 0.0802, Accuracy: 97.53%
Epoch [3/5], Loss: 0.0548, Accuracy: 98.29%
Epoch [4/5], Loss: 0.0393, Accuracy: 98.80%
Epoch [5/5], Loss: 0.0337, Accuracy: 99.00%
Test Accuracy: 98.84%
